# Exercício 1 — ficheiro de texto

In [1]:
with open("inscritos.txt", "w", encoding="utf-8") as ficheiro:
    ficheiro.write("Ana Silva\n")
    ficheiro.write("Bruno Costa\n")
    ficheiro.write("Carla Mendes\n")
    ficheiro.write("Duarte Pinto\n")

with open("inscritos.txt", "r", encoding="utf-8") as ficheiro:
    nomes = [linha.strip() for linha in ficheiro]

print("Número de inscritos:", len(nomes))

for nome in nomes:
    print("-", nome)


Número de inscritos: 4
- Ana Silva
- Bruno Costa
- Carla Mendes
- Duarte Pinto


# Exercício 2 — ficheiro CSV

In [ ]:

import csv


with open("stock.csv", "r", encoding="utf-8") as f:
    leitor = csv.DictReader(f)
    next(leitor)

    mais_caro = None
    menos_stock = None

    for produto in leitor:
        if mais_caro is None or float(produto["preco"]) > float(mais_caro["preco"]):
            mais_caro = produto

        if menos_stock is None or int(produto["quantidade"]) < int(menos_stock["quantidade"]):
            menos_stock = produto

print("Produto mais caro:", mais_caro["produto"])
print("Preço:", mais_caro["preco"], "€")

print("Produto com menos stock:", menos_stock["produto"])
print("Quantidade:", menos_stock["quantidade"])


Produto mais caro: Mochila
Preço: 15.00 €
Produto com menos stock: Mochila
Quantidade: 12


# Exercício 3 — base de dados SQLite

In [ ]:
# Criar base de dados 
import sqlite3

ligacao = sqlite3.connect("cinema.db")
cursor = ligacao.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS filmes (titulo TEXT, classificacao REAL)")
cursor.execute("DELETE FROM filmes")
filmes = [
 ("Matriz", 8.7), ("Robô Desastrado", 6.2),
 ("Viagem ao Fundo do Código", 9.1), ("Loop Infinito", 7.5),
]
cursor.executemany("INSERT INTO filmes VALUES (?, ?)", filmes)
ligacao.commit()
ligacao.close()

In [ ]:
import sqlite3

ligacao = sqlite3.connect("cinema.db")
cursor = ligacao.cursor()
cursor.execute("SELECT titulo, classificacao FROM filmes WHERE classificacao >= 8 ORDER BY classificacao DESC")
for linha in cursor.fetchall():
    print(linha)
ligacao.close()

('Viagem ao Fundo do Código', 9.1)
('Matriz', 8.7)


# Exercício 4 — folha de cálculo XLSX

In [ ]:
# Criar ficheiro Excel
import openpyxl

wb = openpyxl.Workbook(); ws = wb.active; ws.title = "Notas"
ws.append(["nome", "nota"])
for linha in [("Ana", 16), ("Bruno", 12), ("Carla", 18), ("Duarte", 9)]:
 ws.append(linha)
wb.save("notas.xlsx")

In [ ]:
import openpyxl

wb = openpyxl.load_workbook("notas.xlsx")
ws = wb["Notas"]

notas = []

for linha in ws.iter_rows(min_row=2, values_only=True):
    notas.append(linha[1])

media = sum(notas) / len(notas)
print("Média da turma:", round(media, 2))
contador = 0

for nota in notas:
    if nota >= media:
        contador += 1

print("Alunos com nota igual ou superior à média:", contador)
wb.close()

Média da turma: 13.75
Alunos com nota igual ou superior à média: 2


# Exercício 5 — função geradora

In [20]:
def multiplos_de(base, limite):
    numero = base

    while numero <= limite:
        yield numero
        numero += base

for numero in multiplos_de(3, 20):
    print(numero)

print("Soma dos múltiplos de 5:", sum(multiplos_de(5, 100)))


3
6
9
12
15
18
Soma dos múltiplos de 5: 1050


# Atividade de extensão — resumo de vendas (opcional)

In [21]:
import csv
import sqlite3
import openpyxl

ligacao = sqlite3.connect("vendas.db")
cursor = ligacao.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS vendas (
    produto TEXT,
    quantidade INTEGER,
    preco REAL
)
""")

cursor.execute("DELETE FROM vendas")

with open("vendas.csv", "r", encoding="utf-8") as ficheiro:
    leitor = csv.DictReader(ficheiro)

    for linha in leitor:
        cursor.execute(
            "INSERT INTO vendas (produto, quantidade, preco) VALUES (?, ?, ?)",
            (linha["produto"], int(linha["quantidade"]), float(linha["preco"]))
        )

ligacao.commit()

cursor.execute("""
SELECT
    produto,
    SUM(quantidade) AS unidades_vendidas,
    SUM(quantidade * preco) AS valor_total
FROM vendas
GROUP BY produto
ORDER BY produto
""")

resultados = cursor.fetchall()
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Resumo"
ws.append(["produto", "unidades vendidas", "valor total"])

for linha in resultados:
    ws.append([linha[0], linha[1], round(linha[2], 2)])
wb.save("resumo_vendas.xlsx")
ligacao.close()

print("Resumo de vendas criado com sucesso!")


Resumo de vendas criado com sucesso!
